# 04 | Data Preparation, limpieza, feature engineering y pipeline

## Objetivo del notebook

Este notebook explica qué transformaciones se aplican y por qué. La regla principal es:

> **Todo lo que se necesite para predecir en producción debe vivir dentro del pipeline, no solo en el notebook.**

Por eso las variables creadas están implementadas en `src/preprocessing.py` mediante `FeatureEngineeringTransformer`. Así la API puede recibir columnas crudas y el modelo se encarga de preparar el input.


In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)


Project root: /Users/alexandralozano/dp261-g1-final 2


## 1. Principios de limpieza

Las transformaciones se diseñaron con estas reglas:

1. **No usar el target para crear features.** Evita leakage.
2. **No balancear antes del split.** El balanceo solo ocurre dentro de train/folds.
3. **No escalar árboles innecesariamente.** Los modelos de árbol no lo necesitan.
4. **Sí escalar modelos lineales/distancia.** Logistic Regression, SVM y KNN son sensibles a escala.
5. **Conservar columnas originales relevantes.** No se eliminan variables sin justificación.
6. **Encapsular todo en pipeline.** La API debe poder predecir con columnas crudas.


## 2. Variables creadas y justificación

| Feature | Cómo se calcula | Por qué tiene sentido de negocio |
|---|---|---|
| `PurchYear` | Año de `PurchDate` | Puede capturar cambios por periodo o antigüedad del mercado. |
| `PurchMonth` | Mes de `PurchDate` | Puede capturar estacionalidad de subastas o disponibilidad de vehículos. |
| `PurchQuarter` | Trimestre de `PurchDate` | Resume estacionalidad con menos granularidad que mes. |
| `PurchDayOfWeek` | Día de semana de compra | Puede capturar diferencias operativas por día de subasta. |
| `odo_per_year` | `VehOdo / (VehicleAge + 1)` | Mide intensidad de uso anual; un auto joven con mucho kilometraje puede ser riesgoso. |
| `old_high_mileage_flag` | Flag de edad alta y kilometraje alto | Marca vehículos con doble señal de desgaste. |
| `cost_to_acq_auction_avg` | `VehBCost / MMRAcquisitionAuctionAveragePrice` | Detecta si se pagó caro/barato versus mercado al momento de adquisición. |
| `acq_auction_margin` | `MMRAcquisitionAuctionAveragePrice - VehBCost` | Mide colchón económico frente al valor de subasta. |
| `cost_to_current_auction_avg` | `VehBCost / MMRCurrentAuctionAveragePrice` | Compara costo con valor actual de subasta. |
| `current_auction_margin` | `MMRCurrentAuctionAveragePrice - VehBCost` | Mide margen actual esperado en mercado de subasta. |
| `warranty_to_cost` | `WarrantyCost / VehBCost` | Garantía alta respecto al costo puede indicar mayor riesgo mecánico. |
| `warranty_per_vehicle_year` | `WarrantyCost / (VehicleAge + 1)` | Ajusta costo de garantía por antigüedad. |
| `auction_avg_depreciation` | `MMRCurrentAuctionAveragePrice - MMRAcquisitionAuctionAveragePrice` | Captura cambio de valor de mercado entre adquisición y momento actual. |
| `retail_avg_depreciation` | `MMRCurrentRetailAveragePrice - MMRAcquisitionRetailAveragePrice` | Captura cambio de valor esperado en retail. |
| `acq_clean_avg_spread` | `MMRAcquisitionAuctionCleanPrice - MMRAcquisitionAuctionAveragePrice` | Diferencia entre condición clean y promedio al adquirir. |
| `current_clean_avg_spread` | `MMRCurrentAuctionCleanPrice - MMRCurrentAuctionAveragePrice` | Diferencia actual entre condición clean y promedio. |
| `mmr_missing_count` | Conteo de MMR nulos | La falta de información de mercado puede ser una señal de incertidumbre/riesgo. |

Aunque el profesor menciona “10 variables” como ejemplo, aquí se dejan más de 10 porque varias son familias coherentes: calendario, desgaste, costo vs mercado, garantía, depreciación y calidad de información.


In [2]:
import pandas as pd
from src.config import RAW_DATA_PATH, TARGET
from src.preprocessing import prepare_features, split_X_y, build_preprocessor

df = pd.read_csv(RAW_DATA_PATH)
X, y = split_X_y(df)
X_fe = prepare_features(X)
print('Columnas originales sin target:', X.shape[1])
print('Columnas luego de feature engineering:', X_fe.shape[1])
X_fe.head()


Columnas originales sin target: 32
Columnas luego de feature engineering: 48


,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,WheelTypeID,...,acq_auction_margin,cost_to_current_auction_avg,current_auction_margin,warranty_to_cost,warranty_per_vehicle_year,auction_avg_depreciation,retail_avg_depreciation,acq_clean_avg_spread,current_clean_avg_spread,mmr_missing_count
0,ADESA,2006,3,MAZDA,MAZDA3,I,4D SEDAN I,RED,AUTO,1,...,1055.0,0.952892,351.0,0.156761,278.25,-704.0,-39.0,1674.0,1101.0,0
1,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,WHITE,AUTO,1,...,-746.0,1.019313,-144.0,0.138553,175.50,602.0,477.0,1529.0,1766.0,0
2,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,MAROON,AUTO,2,...,-1698.0,1.214374,-865.0,0.283469,277.80,833.0,203.0,1558.0,1522.0,0
3,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,SILVER,AUTO,1,...,-2207.0,2.223427,-2256.0,0.153659,105.00,-49.0,-283.0,782.0,802.0,0
4,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,SILVER,MANUAL,2,...,-87.0,1.231906,-753.0,0.255000,204.00,-666.0,-984.0,1141.0,1137.0,0


## 3. Tratamiento de nulos y atípicos

El pipeline aplica:

- Numéricos: imputación por mediana, porque es robusta ante colas largas y outliers.
- Categóricos: imputación con `MISSING`, porque la ausencia puede ser informativa.
- MMR <= 0: se convierte a nulo porque un precio de mercado cero no es económicamente válido.
- Texto: se normaliza a mayúsculas y se eliminan espacios para no duplicar categorías.

No se eliminan registros masivamente porque perderíamos casos de la clase minoritaria.


## 4. Pipelines diferenciados por familia de modelo

| Familia | Preprocesamiento | Balanceo |
|---|---|---|
| Logistic Regression / SVM / KNN | Imputación + escalado + OneHotEncoding | `RandomUnderSampler` dentro del pipeline + `class_weight` cuando aplica |
| Decision Tree / Random Forest / Bagging | Imputación + OneHotEncoding, sin escalado obligatorio | `class_weight='balanced'` o `balanced_subsample` |
| XGBoost / LightGBM | Imputación + OrdinalEncoder | `scale_pos_weight` para clase minoritaria |

Se evita SMOTE como primera opción porque hay muchas variables categóricas codificadas. SMOTE sobre OHE puede crear combinaciones sintéticas difíciles de interpretar, por ejemplo medias entre categorías de marca/modelo. Por eso se usa submuestreo controlado en modelos lineales/distancia y pesos de clase en árboles/boosting.


In [3]:
preproc_linear = build_preprocessor(X.head(100), mode='linear')
preproc_tree = build_preprocessor(X.head(100), mode='tree_ohe')
preproc_ordinal = build_preprocessor(X.head(100), mode='ordinal')
print(preproc_linear)


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', RobustScaler())]),
                                 ['VehYear', 'VehicleAge', 'VehOdo',
                                  'MMRAcquisitionAuctionAveragePrice',
                                  'MMRAcquisitionAuctionCleanPrice',
                                  'MMRAcquisitionRetailAveragePrice',
                                  'MMRAcquisitonRetailCleanPrice',
                                  'MMRCurrentAuctionAveragePrice',
                                  'MMRCurrentAuctionCl...
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='MISSING',
                                                                strategy='con

/Users/alexandralozano/dp261-g1-final 2/src/preprocessing.py:30: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"NAN": np.nan, "NONE": np.nan, "<NA>": np.nan, "": np.nan})
/Users/alexandralozano/dp261-g1-final 2/src/preprocessing.py:30: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"NAN": np.nan, "NONE": np.nan, "<NA>": np.nan, "": np.nan})
/Users/alexandralozano/dp261-g1-final 2/src/preprocessing.py:30: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To re

## 5. Validación de no leakage

El objeto final de modelado siempre tiene esta estructura:

```text
FeatureEngineeringTransformer -> ColumnTransformer -> sampler opcional -> clasificador
```

Como todo está dentro del pipeline, `cross_validate`, `RandomizedSearchCV` y `Optuna` ajustan transformaciones solo en el fold de entrenamiento. El fold de validación se transforma con parámetros aprendidos en train.


In [4]:
from src.models import build_baseline_models
models = build_baseline_models(X.head(500))
models.keys()


dict_keys(['LogisticRegression_baseline', 'DecisionTree_baseline', 'RandomForest_baseline', 'SVM_baseline', 'KNN_baseline'])